In [115]:
import os 
import pandas as pd 
from pathlib import Path 
from dataclasses import dataclass


In [116]:
@dataclass
class DataPreProcessingConfig:
    root_dir : Path 
    data_path : Path 
    local_data_file : str 

In [117]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification'

In [118]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [119]:
from ServicenowGamification.Utils.Common import * 
from ServicenowGamification.Constant import * 
from ServicenowGamification.Config import *

In [120]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [121]:
class ConfigurationManager:
    def __init__(self,
                 CONFIG_PATH=CONFIG_FILE_PATH,
                 PARAMS_PATH=PARAMS_FILE_PATH
                 ):
         print(CONFIG_PATH)
                 
         print(f"path config yaml {Path(CONFIG_PATH)}")
         self.config=Read_Yaml(Path(CONFIG_PATH))
         self.params=Read_Yaml(Path(PARAMS_PATH))
         self.Data_PreProcessing = self.config.Data_PreProcessing
    def get_datapreprocessing(self)->DataPreProcessingConfig:
         self.config=self.Data_PreProcessing
         data_preprocessingconfig=DataPreProcessingConfig(
              root_dir=self.config.root_dir,
              data_path=self.config.data_path,
              local_data_file=self.config.local_data_file
         )
         return data_preprocessingconfig


In [ ]:
class DataPreProcessing:
    def __init__(self,config:DataPreProcessingConfig):

        self.config=config

        create_directories([self.config.root_dir])

    def Datapreprocessing(self):
        df=pd.read_csv(self.config.data_path)
        #print(df.head(2))
        #print(df.shape)           
        features=[
            "number",
            "opened_at",
            "resolved_at",
            "closed_at",
            "priority",
            "state",
            "assigned_to",
            "assignment_group",
            "category",
            "subcategory",
            "short_description",
            "description",
            "sys_created_on",
            "sys_updated_on",
        ]
        dp=df[features]
        #print(dp.head(1))

        print(f"\n*----------------- Remove duplicates from Incident Data Frame -----------------*\n")
        print(f" Data Frame Shape before Remove duplicates {dp.shape}")
        dp=dp.drop_duplicates(subset=["number"])
        print(f" Data Frame Shape after Remove duplicatess {dp.shape}")

        print(f"\n*-------- Date Features: Convert to datetime from Incident Data Frame ---------*\n")
        # Convert to datetime
        dp["opened_at"] = pd.to_datetime(dp["opened_at"], errors="coerce")
        dp["resolved_at"] = pd.to_datetime(dp["resolved_at"], errors="coerce")
        dp["closed_at"] = pd.to_datetime(dp["closed_at"], errors="coerce")
        dp["sys_created_on"] = pd.to_datetime(dp["sys_created_on"], errors="coerce")
        dp["sys_updated_on"] = pd.to_datetime(dp["sys_updated_on"], errors="coerce")

        print(f"\n*-------------- Handle missing values from Incident Data Frame ----------------*\n")
        print(f"Checking Missing Values count for every object features  before Handle missing values ")
       
        object_columns = dp.select_dtypes(include=['object', 'string']).columns
        for column in object_columns:
            if dp[column].isnull().sum()>0:
                fill_values = {

                    "assigned_to": ["Venakt", "Vamsi", "Raju", "Ganesh", "Chaitanya"],
                    "assignment_group": ["Network", "Database", "Data", "QA", "Cloud"],
                    "category": ["Hardware", "Software", "Network", "Access", "Email"],
                    "subcategory": ["Laptop", "Application", "VPN", "Password", "Outlook"],

                    "description": [
                        "User reported an issue",
                        "Application not responding",
                        "Unable to login",
                        "System performance issue",
                        "Email issue reported"]
                    }

                for column in object_columns:
                    if dp[column].isnull().sum()>0:
                        print(f"   {column }- Missing count :{dp[column].isnull().sum()} ")

                
                for column, values in fill_values.items():
                    if column in object_columns:
                        missing_index = dp[dp[column].isna()].index

                        for i, idx in enumerate(missing_index):
                            dp.at[idx, column] = values[(i // 5) % len(values)]
                print(f"Checking Missing Values count for every object features  After Handle missing values ")
                if dp[column].isnull().sum()>0:
                    for column in object_columns:
                        if dp[column].isnull().sum()>0:
                            print(f"  {column }- Missing count :{dp[column].isnull().sum()} ")
                else :
                    print(f"No Missing values ")
        print(f"\nChecking Missing Values count for Date features before Handle missing values ")
        date_columns = dp.select_dtypes(include=['datetime64[ns]', 'datetimetz']).columns
        for column in date_columns:
            if dp[column].isnull().sum()>0:
                print(f"  {column }- Missing count :{dp[column].isnull().sum()} ")

        for dt_column in date_columns:

            if dt_column=='resolved_at':
                print(f"\n{dt_column}-Handle missing values based on opened_at ")    
                for idx in dp[dp["resolved_at"].isna()].index:
                    days = (idx % 5) + 1      # 1,2,3,4,5 days
                    dp.at[idx, "resolved_at"] = (
                        dp.at[idx, "opened_at"] + pd.Timedelta(days=days)
                    )
        for dt_column in date_columns:
                    if dt_column=='closed_at':
                        print(f"\n{dt_column}-Handle missing values based on closed_at ")     
                        for idx in dp[dp["closed_at"].isna()].index:
                            dp.at[idx, "closed_at"] = (
                                dp.at[idx, "resolved_at"] + pd.Timedelta(days=1)
                            )
        
#         print(f"\n*-------------- Calculate MTTR(Mean Time To Resolve) ----------------*\n")
#         date_columns = ["opened_at", "resolved_at", "closed_at"]

#         for col in date_columns:
#             dp[col] = pd.to_datetime(dp[col], errors="coerce")
#         print(dp[date_columns].dtypes)    
#         #Resolution Time = Resolved Time - Opened Time
#         #df["resolution_time"] = (dp["resolved_at"] - dp["opened_at"])
#         dp["resolution_time"] = dp["resolved_at"] - dp["opened_at"]
#         # Convert to hours:
#         #dp["resolution_hours"] = (dp["resolution_time"].dt.total_seconds() / 3600)
#         #dp["resolution_hours"] = (dp["resolution_time"].dt.total_seconds() / 3600)
#         dp["resolution_hours"] = ((dp["resolved_at"] - dp["opened_at"]).dt.total_seconds() / 3600
# )
       
#         print(f"\n*--------------  overall MTTR (Mean Time To Resolve) ----------------*\n")
#        # print(dp[["number", "resolution_hours"]].head())
#         mttr = dp["resolution_hours"].mean()
#         print("MTTR:", mttr)

#         print(dp[["number", "resolution_hours"]].head(1))        




        
            

In [145]:
config = ConfigurationManager()

data_preprocessing_config = config.get_datapreprocessing()

data_PreProcess = DataPreProcessing(data_preprocessing_config)   # ✅ Correct
df=data_PreProcess.Datapreprocessing()


Config\Config.yaml
path config yaml Config\Config.yaml
Reading YAML : Config\Config.yaml
[2026-08-01 12:05:07,005: INFO: Common: YAML file loaded successfully : Config\Config.yaml]
Reading YAML : params.yaml
[2026-08-01 12:05:07,019: INFO: Common: YAML file loaded successfully : params.yaml]
[2026-08-01 12:05:07,023: INFO: Common: Created directory : Artifacts/DataPreProcessing]

*----------------- Remove duplicates from Incident Data Frame -----------------*

 Data Frame Shape before Remove duplicates (67, 14)
 Data Frame Shape after Remove duplicatess (67, 14)

*-------- Date Features: Convert to datetime from Incident Data Frame ---------*


*-------------- Handle missing values from Incident Data Frame ----------------*

Checking Missing Values count for every object features  before Handle missing values 
   category- Missing count :4 
   subcategory- Missing count :64 
   description- Missing count :4 
Checking Missing Values count for every object features  After Handle missing 